<a href="https://colab.research.google.com/github/Cheetah-lhp/MachineLearning/blob/main/DL_MOE_cat_lgb_rf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install catboost
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 16.3 MB/s eta 0:00:00


In [3]:
import pickle
import ast
import optuna
import pandas as pd
import numpy as np

import lightgbm as lgb
from catboost import CatBoostClassifier, Pool
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder

In [4]:
trainDataset = pickle.load(open('/content/train.pkl', 'rb'))
testDataset = pickle.load(open('/content/test.pkl', 'rb'))

#Xử lý dữ liệu thô trước khi đưa vào mô hình

In [5]:
def extract_feature(values):
    if values is None or isinstance(values, (int, float)):
        vals = values if isinstance(values, (int, float)) else np.nan
        return {'mean': vals, 'max': vals, 'min': vals, 'std': 0,
                'last': vals, 'first': vals, 'count': 0 if np.isnan(vals) else 1, 'trend': 0}

    if len(values) == 0:
        return {'mean': np.nan, 'max': np.nan, 'min': np.nan, 'std': np.nan,
                'last': np.nan, 'first': np.nan, 'count': 0, 'trend': np.nan}

    vals = [v['value'] for v in values if v['value'] is not None]
    if len(vals) == 0:
        return {'mean': np.nan, 'max': np.nan, 'min': np.nan, 'std': np.nan,
                'last': np.nan, 'first': np.nan, 'count': 0, 'trend': np.nan}

    return {
        'mean': np.mean(vals),
        'max': np.max(vals),
        'min': np.min(vals),
        'std': np.std(vals) if len(vals) > 1 else 0, # Tránh lỗi chia 0
        'last': vals[-1],
        'first': vals[0],           # Thêm đặc trưng đo lần đầu
        'count': len(vals),         # Thêm đặc trưng số lần đo
        'trend': vals[-1] - vals[0] # Thêm đặc trưng xu hướng (tăng/giảm)
    }

def process_dataset(data):
    rows = []

    # Các phím đặc biệt không phải là chỉ số đo đạc thông thường
    special_keys = ['target', 'specimen', 'age_at_admission', 'gender', 'race']

    for pid, patient in data.items():
        row = {}

        # 1. Tự động lấy tất cả các đặc tính đo đạc có trong dữ liệu của bệnh nhân này
        all_features = [k for k in patient.keys() if k not in special_keys]

        for feat in all_features:
            # Trích xuất 8 chỉ số (mean, max, min, std, last, first, count, trend)
            stats = extract_feature(patient.get(feat, []))
            for k, v in stats.items():
                row[f"{feat}_{k}"] = v

        # 2. Xử lý các cột đặc biệt (giữ nguyên logic của bạn)
        specimen = patient.get('specimen', [])
        row['specimen'] = specimen[-1]['value'] if (isinstance(specimen, list) and len(specimen) > 0) else 'unknown'

        row['age'] = patient.get('age_at_admission', np.nan)
        row['gender'] = patient.get('gender', 'unknown')
        row['race'] = patient.get('race', 'unknown')
        row['target'] = patient.get('target', 0)

        rows.append(row)

    return pd.DataFrame(rows)

df_train = process_dataset(trainDataset)

df_test = process_dataset(testDataset)

#Stack Learning

##**Phase 1a:**
- train model để CatBoost học được insigt dữ liệu

In [6]:
# 1. Tiền xử lý dữ liệu
print("Đang chuẩn bị dữ liệu cho CatBoost...")

# Xác định danh sách các cột phân loại (categorical features)
# CatBoost sẽ tự xử lý các cột này, không cần One-Hot Encoding
cat_features = ['specimen', 'gender', 'race']

X = df_train.drop(columns=['target'])
y = df_train['target']

for column in cat_features:
    if column in X.columns:
        X[column] = X[column].fillna('Missing').astype(str)

# 2. Chia tập dữ liệu (80% train, 20% val)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Khởi tạo và huấn luyện CatBoost
# Chúng ta sử dụng cấu trúc Pool để CatBoost hiểu rõ đâu là biến phân loại
train_pool = Pool(X_train, y_train, cat_features=cat_features)
val_pool = Pool(X_val, y_val, cat_features=cat_features)

model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.03,
    depth=6,
    auto_class_weights='Balanced',
    eval_metric='AUC',
    random_seed=42,
    verbose=100,
    l2_leaf_reg=5.0,
    subsample=0.8
)

model.fit(train_pool, eval_set=val_pool, early_stopping_rounds=100)

# 4. Dự báo xác suất và tính ROC AUC
y_probs = model.predict_proba(X_val)[:, 1]
roc_auc = roc_auc_score(y_val, y_probs)

print("-" * 30)
print(f"Kết quả CatBoost:")
print(f"ROC AUC Score: {roc_auc:.4f}")
print("-" * 30)

Đang chuẩn bị dữ liệu cho CatBoost...
0:	test: 0.7707215	best: 0.7707215 (0)	total: 166ms	remaining: 2m 46s
100:	test: 0.8542213	best: 0.8545062 (98)	total: 11.8s	remaining: 1m 44s
200:	test: 0.8617721	best: 0.8617721 (199)	total: 24.2s	remaining: 1m 36s
300:	test: 0.8667020	best: 0.8667020 (300)	total: 34.1s	remaining: 1m 19s
400:	test: 0.8703663	best: 0.8703910 (398)	total: 44.2s	remaining: 1m 6s
500:	test: 0.8709924	best: 0.8712818 (441)	total: 54s	remaining: 53.8s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.8712818188
bestIteration = 441

Shrink model to first 442 iterations.
------------------------------
Kết quả CatBoost:
ROC AUC Score: 0.8713
------------------------------


#**Phase 1b**:
- Từ model CatBoost đã học được insight data, dùng *FEATURE IMPORTANCE* có sẵn trên CatBoost để lọc ra các feature có ảnh hưởng lớn nhất đến kết quả

- Dùng  lib *OPTUNA* để tìm hyparameters cho prediction (model, training, ..)

In [7]:
# ==========================================
# CHỌN LỌC TOP 150 ĐẶC TÍNH (Từ model_filtered trước đó)
# ==========================================
print("1. Đang chọn lọc Top 150 đặc tính quan trọng nhất...")
importance = model.get_feature_importance()
feature_importances = pd.DataFrame({'feature': X_train.columns, 'importance': importance})
feature_importances = feature_importances.sort_values(by='importance', ascending=False)
top_150_features = feature_importances['feature'].head(150).tolist()

for cat in cat_features:
    if cat in X_train.columns and cat not in top_150_features:
        top_150_features.append(cat)

X_train_top = X_train[top_150_features]
X_val_top = X_val[top_150_features]
current_cat_features = [c for c in cat_features if c in X_train_top.columns]

# ==========================================
# TÌM SIÊU THAM SỐ VỚI OPTUNA (Trên 1 tập Val để chạy nhanh)
# ==========================================
print("\n2. Khởi động Optuna để tìm siêu tham số tối ưu...")
def objective(trial):
    params = {
        'iterations': 500, # Chạy ít vòng để dò tìm cho nhanh
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'depth': trial.suggest_int('depth', 4, 8),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 1, 50),
        'auto_class_weights': 'Balanced',
        'eval_metric': 'AUC', 'random_seed': 42, 'verbose': 0
    }
    opt_model = CatBoostClassifier(**params)
    opt_model.fit(Pool(X_train_top, y_train, cat_features=current_cat_features),
                  eval_set=Pool(X_val_top, y_val, cat_features=current_cat_features),
                  early_stopping_rounds=50)
    return roc_auc_score(y_val, opt_model.predict_proba(X_val_top)[:, 1])

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)
best_params = study.best_params

print(f"\n-> Optuna đã tìm xong! ROC AUC tốt nhất: {study.best_value:.4f}")

# ==========================================
# HUẤN LUYỆN 5-FOLD VỚI BỘ THAM SỐ TỪ OPTUNA
# ==========================================
print("\n3. Bắt đầu huấn luyện Stratified 5-Fold với bộ tham số từ Optuna...")
best_params['iterations'] = 1500 # Trả lại số vòng lặp lớn cho quá trình train thật
best_params['auto_class_weights'] = 'Balanced'
best_params['eval_metric'] = 'AUC'
best_params['verbose'] = 200

# Gộp chung X_train và X_val lại để chia 5-Fold cho chuẩn
X_full_top = pd.concat([X_train_top, X_val_top]).reset_index(drop=True)
y_full = pd.concat([y_train, y_val]).reset_index(drop=True)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cat_oof_preds = np.zeros(len(X_full_top))
fold_scores = []
catboost_models = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_full_top, y_full)):
    print(f"\n--- Đang huấn luyện Fold {fold + 1}/5 ---")
    best_params['random_seed'] = 42 + fold # Đổi seed để mô hình đa dạng hơn

    X_f_train, X_f_val = X_full_top.iloc[train_idx], X_full_top.iloc[val_idx]
    y_f_train, y_f_val = y_full.iloc[train_idx], y_full.iloc[val_idx]

    train_pool = Pool(X_f_train, y_f_train, cat_features=current_cat_features)
    val_pool = Pool(X_f_val, y_f_val, cat_features=current_cat_features)

    model_cv = CatBoostClassifier(**best_params)
    model_cv.fit(train_pool, eval_set=val_pool, early_stopping_rounds=100)

    val_probs = model_cv.predict_proba(X_f_val)[:, 1]
    cat_oof_preds[val_idx] = val_probs
    fold_auc = roc_auc_score(y_f_val, val_probs)

    fold_scores.append(fold_auc)
    catboost_models.append(model_cv)
    print(f"-> Điểm Fold {fold + 1}: {fold_auc:.4f}")

print("=" * 50)
print("KẾT QUẢ TỔNG HỢP (TOP 150 + OPTUNA + 5-FOLD):")
print(f"ROC AUC OOF tổng quát: {roc_auc_score(y_full, cat_oof_preds):.4f}")
print("=" * 50)

[I 2026-05-09 05:11:35,072] A new study created in memory with name: no-name-81ccf1f7-8b33-4b34-9b91-756d2d962fca


1. Đang chọn lọc Top 150 đặc tính quan trọng nhất...

2. Khởi động Optuna để tìm siêu tham số tối ưu...


[I 2026-05-09 05:12:24,191] Trial 0 finished with value: 0.8696796574875237 and parameters: {'learning_rate': 0.031282173353273414, 'depth': 8, 'l2_leaf_reg': 5.1155265818974565, 'subsample': 0.6012673244010659, 'min_data_in_leaf': 9}. Best is trial 0 with value: 0.8696796574875237.
[I 2026-05-09 05:12:45,956] Trial 1 finished with value: 0.8716991885972786 and parameters: {'learning_rate': 0.038012033281185055, 'depth': 5, 'l2_leaf_reg': 1.434220613163489, 'subsample': 0.8441043706354031, 'min_data_in_leaf': 29}. Best is trial 1 with value: 0.8716991885972786.
[I 2026-05-09 05:13:04,326] Trial 2 finished with value: 0.8682772053279719 and parameters: {'learning_rate': 0.05417367645976591, 'depth': 7, 'l2_leaf_reg': 3.7286596709431716, 'subsample': 0.9651180774915092, 'min_data_in_leaf': 4}. Best is trial 1 with value: 0.8716991885972786.
[I 2026-05-09 05:13:17,105] Trial 3 finished with value: 0.8673392453236636 and parameters: {'learning_rate': 0.02712401519892742, 'depth': 5, 'l2_le


-> Optuna đã tìm xong! ROC AUC tốt nhất: 0.8725

3. Bắt đầu huấn luyện Stratified 5-Fold với bộ tham số từ Optuna...

--- Đang huấn luyện Fold 1/5 ---
0:	test: 0.7859073	best: 0.7859073 (0)	total: 41.3ms	remaining: 1m 1s
200:	test: 0.8706872	best: 0.8706872 (200)	total: 7.4s	remaining: 47.8s
400:	test: 0.8747195	best: 0.8748070 (398)	total: 13.6s	remaining: 37.4s
600:	test: 0.8750381	best: 0.8756620 (506)	total: 21s	remaining: 31.4s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.8756619574
bestIteration = 506

Shrink model to first 507 iterations.
-> Điểm Fold 1: 0.8757

--- Đang huấn luyện Fold 2/5 ---
0:	test: 0.7991296	best: 0.7991296 (0)	total: 35.6ms	remaining: 53.4s
200:	test: 0.8855105	best: 0.8855981 (199)	total: 6.72s	remaining: 43.5s
400:	test: 0.8875615	best: 0.8883244 (354)	total: 14.2s	remaining: 39.1s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.8883244175
bestIteration = 354

Shrink model to first 355 iterations.
-> Điểm Fol

#**Phase 2**: Dùng các feature tốt nhất từ phase 1 để predict cho LGBoost và RandomForest ở phase 2

In [8]:
print("1. Chuẩn bị dữ liệu cho LightGBM...")
# LightGBM yêu cầu các cột phân loại phải có định dạng 'category' của Pandas
X_lgb_full = X_full_top.copy()
for col in current_cat_features:
    X_lgb_full[col] = X_lgb_full[col].astype('category')

# Thiết lập tham số cho LightGBM
lgb_params = {
    'objective': 'binary',
    'metric': 'auc',
    'learning_rate': 0.03,
    'max_depth': 6,
    'num_leaves': 31,              # Điểm đặc trưng của LGBM (lá)
    'class_weight': 'balanced',    # Xử lý mất cân bằng lớp
    'feature_fraction': 0.8,       # Tương đương subsample cột
    'bagging_fraction': 0.8,       # Tương đương subsample hàng
    'bagging_freq': 1,
    'verbose': -1,                 # Tắt cảnh báo
    'random_state': 42
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
lgb_oof_preds = np.zeros(len(X_lgb_full))

print("\n2. Bắt đầu huấn luyện LightGBM (5-Fold)...")
lgb_models = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_lgb_full, y_full)):
    X_train_f, y_train_f = X_lgb_full.iloc[train_idx], y_full.iloc[train_idx]
    X_val_f, y_val_f = X_lgb_full.iloc[val_idx], y_full.iloc[val_idx]

    # Tạo dataset chuẩn của LightGBM
    train_data = lgb.Dataset(X_train_f, label=y_train_f, categorical_feature=current_cat_features)
    val_data = lgb.Dataset(X_val_f, label=y_val_f, categorical_feature=current_cat_features, reference=train_data)

    # Huấn luyện
    model_lgb = lgb.train(
        lgb_params,
        train_data,
        num_boost_round=1500,
        valid_sets=[val_data],
        callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False),
                   lgb.log_evaluation(period=0)] # Tắt log để giao diện gọn gàng
    )

    # Dự đoán
    lgb_oof_preds[val_idx] = model_lgb.predict(X_val_f, num_iteration=model_lgb.best_iteration)
    lgb_models.append(model_lgb)
    print(f"-> LightGBM Fold {fold + 1} ROC AUC: {roc_auc_score(y_val_f, lgb_oof_preds[val_idx]):.4f}")

lgb_auc = roc_auc_score(y_full, lgb_oof_preds)
print(f"\n=> ĐIỂM TỔNG QUÁT CỦA ĐỘC LẬP LIGHTGBM: {lgb_auc:.4f}")


1. Chuẩn bị dữ liệu cho LightGBM...

2. Bắt đầu huấn luyện LightGBM (5-Fold)...
-> LightGBM Fold 1 ROC AUC: 0.8743
-> LightGBM Fold 2 ROC AUC: 0.8921
-> LightGBM Fold 3 ROC AUC: 0.8847
-> LightGBM Fold 4 ROC AUC: 0.8803
-> LightGBM Fold 5 ROC AUC: 0.8988

=> ĐIỂM TỔNG QUÁT CỦA ĐỘC LẬP LIGHTGBM: 0.8852


In [9]:
print("1. Chuẩn bị dữ liệu cho Random Forest...")

# RandomForest của sklearn KHÔNG hỗ trợ dtype='category'
# => cần encode categorical features sang số
# Copy dữ liệu
X_rf_full = X_full_top.copy()

# Encode categorical columns
label_encoders = {}

for col in current_cat_features:
    le = LabelEncoder()

    # convert sang string để tránh lỗi NaN/category
    X_rf_full[col] = X_rf_full[col].astype(str)
    X_rf_full[col] = le.fit_transform(X_rf_full[col])
    label_encoders[col] = le

rf_params = {
    'n_estimators': 500,
    'max_depth': 10,
    'min_samples_split': 10,
    'min_samples_leaf': 5,
    'max_features': 'sqrt',
    'class_weight': 'balanced',
    'random_state': 42,
    'n_jobs': -1
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rf_oof_preds = np.zeros(len(X_rf_full))

print("\n2. Bắt đầu huấn luyện Random Forest (5-Fold)...")

rf_models = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_rf_full, y_full)):

    X_train_f = X_rf_full.iloc[train_idx]
    y_train_f = y_full.iloc[train_idx]

    X_val_f = X_rf_full.iloc[val_idx]
    y_val_f = y_full.iloc[val_idx]

    # Khởi tạo model
    model_rf = RandomForestClassifier(**rf_params)

    # Huấn luyện
    model_rf.fit(X_train_f, y_train_f)

    # Predict probability cho class 1
    rf_oof_preds[val_idx] = model_rf.predict_proba(X_val_f)[:, 1]

    rf_models.append(model_rf)

    fold_auc = roc_auc_score(y_val_f, rf_oof_preds[val_idx])

    print(f"-> Random Forest Fold {fold + 1} ROC AUC: {fold_auc:.4f}")

# Điểm tổng
rf_auc = roc_auc_score(y_full, rf_oof_preds)

print(f"\n=> ĐIỂM TỔNG QUÁT CỦA RANDOM FOREST: {rf_auc:.4f}")

1. Chuẩn bị dữ liệu cho Random Forest...

2. Bắt đầu huấn luyện Random Forest (5-Fold)...
-> Random Forest Fold 1 ROC AUC: 0.8594
-> Random Forest Fold 2 ROC AUC: 0.8853
-> Random Forest Fold 3 ROC AUC: 0.8767
-> Random Forest Fold 4 ROC AUC: 0.8616
-> Random Forest Fold 5 ROC AUC: 0.8838

=> ĐIỂM TỔNG QUÁT CỦA RANDOM FOREST: 0.8731


#**Mixture of Experts (MoE)** - Deep Learning.

In [11]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Softmax, Multiply, Lambda
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

In [15]:
print("\n3. Training Deep Learning Mixture of Experts (MoE)...")

#meta feature từ các experts (CatBoost, LGBoost, RF)
meta_features = np.column_stack([
    cat_oof_preds,
    lgb_oof_preds,
    rf_oof_preds
])

# =========================================================
# MoE Model
# =========================================================

def moe_model(num_experts):
  inputs = Input(shape=(num_experts,))

  #gating network
  gate = Dense(16, activation='relu')(inputs)
  gate = Dense(8, activation='relu')(gate)
  gate = Dense(num_experts)(gate)

  gate_weights = Softmax(name='gate_softmax')(gate)

  #weight expert outputs
  weighted_experts = Multiply()([inputs, gate_weights])

  output = Lambda(lambda x: tf.reduce_sum(x, axis=1, keepdims=True))(weighted_experts)

  model = Model(inputs=inputs, outputs=output)
  model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy')

  return model

# =========================================================
# Cross Validation
# =========================================================
meta_skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
moe_oof_preds = np.zeros(len(meta_features))
moe_models = []

for fold, (train_idx, val_idx) in enumerate(meta_skf.split(meta_features, y_full)):
  X_train_meta = meta_features[train_idx]
  y_train_meta = y_full.iloc[train_idx]
  X_val_meta = meta_features[val_idx]
  y_val_meta = y_full.iloc[val_idx]

  model_moe = moe_model(num_experts=3)

  early_stop = tf.keras.callbacks.EarlyStopping(
      monitor='val_loss',
      patience=20,
      restore_best_weights=True
  )

  model_moe.fit(
      X_train_meta,
      y_train_meta,
      validation_data=(X_val_meta, y_val_meta),
      epochs=200,
      batch_size=256,
      callbacks=[early_stop],
      verbose=0
  )

  preds = model_moe.predict(X_val_meta, verbose=0).flatten()

  moe_oof_preds[val_idx] = preds
  moe_models.append(model_moe)
  fold_auc = roc_auc_score(y_val_meta, preds)
  print(f"-> MoE Fold {fold+1} ROC AUC: {fold_auc:.4f}")

#final pred
moe_auc = roc_auc_score(y_full, moe_oof_preds)
print("=" * 60)
print(f"FINAL MOE ROC AUC: {moe_auc:.4f}")
print("=" * 60)


3. Training Deep Learning Mixture of Experts (MoE)...
-> MoE Fold 1 ROC AUC: 0.8762
-> MoE Fold 2 ROC AUC: 0.8927
-> MoE Fold 3 ROC AUC: 0.8867
-> MoE Fold 4 ROC AUC: 0.8800
-> MoE Fold 5 ROC AUC: 0.9007
FINAL MOE ROC AUC: 0.8873


In [17]:
X_test_top = df_test.reindex(columns=top_150_features, fill_value=np.nan).copy()
for column in current_cat_features:
    X_test_top[column] = X_test_top[column].astype(str)

#process test data
# LightGBM cần category dtype
X_test_lgb = X_test_top.copy()
for col in current_cat_features:
    X_test_lgb[col] = X_test_lgb[col].astype('category')

# RandomForest cần label encoding
X_test_rf = X_test_top.copy()

for col in current_cat_features:
    X_test_rf[col] = X_test_rf[col].astype(str)
    known_classes = set(label_encoders[col].classes_)
    X_test_rf[col] = X_test_rf[col].apply(
        lambda x: x if x in known_classes else 'UNK'
    )
    if 'UNK' not in label_encoders[col].classes_:
        label_encoders[col].classes_ = np.append(
            label_encoders[col].classes_,
            'UNK'
        )
    X_test_rf[col] = label_encoders[col].transform(
        X_test_rf[col]
    )

#experts model prediction
#LGB
lgb_test_preds = np.zeros(len(X_test_lgb))

for lgb_model in lgb_models:
    lgb_test_preds += (
        lgb_model.predict(X_test_lgb) / len(lgb_models)
    )

#CatBoost
cat_test_preds = np.zeros(len(X_test_top))
for model in catboost_models:
    cat_test_preds += model.predict_proba(X_test_top)[:, 1] / len(catboost_models)

#RF
rf_test_preds = np.zeros(len(X_test_rf))

for rf_model in rf_models:

    rf_test_preds += (
        rf_model.predict_proba(X_test_rf)[:, 1] / len(rf_models)
    )

#MOE meta model
meta_test_features = np.column_stack([
    cat_test_preds,
    lgb_test_preds,
    rf_test_preds
])

#MOE prediction
prob = np.zeros(len(meta_test_features))
for moe_model in moe_models:
    prob += (
        moe_model.predict(
            meta_test_features,
            verbose=0
        ).flatten() / len(moe_models)
    )
pred = (prob >= 0.5).astype(int)

In [ ]:
testDataset = "/content/test.pkl"
with open(testDataset, 'rb') as f:
    dataTest = pickle.load(f)

patient_ids_list = list(dataTest.keys())

submission = pd.DataFrame({
    'id': patient_ids_list,
    'probability': prob,
    'prediction': pred
})

submission['probability'] = submission['probability'].round(4)

predFile = "group15.csv"
submission.to_csv(predFile, index=False)

submission.head()

,id,probability,prediction
0,30719950,0.1804,0
1,31639067,0.2226,0
2,34436833,0.3314,0
3,30027612,0.0557,0
4,39655034,0.5549,1
